In [177]:
import json
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, field_validator
import instructor
import os
from datetime import datetime
from neo4j import GraphDatabase
from groq import Groq

In [3]:
maildir = "maildir"

# Data extraction and structuring using LLM

In [ ]:
# this is just my username and password
username, pwd = (None, None)
groq_api_key = None
gemini_api_key = None
driver = GraphDatabase.driver("bolt://localhost:7687", auth=(username, pwd))

In [190]:
# classes for structuring our data

ALLOWED_PREDICATES = {"WORKS_ON", "WORKS_UNDER", "CAUSES_ISSUE_IN", "RESOLVED", "NOT_RESOLVED", "MANAGES", "LOCATED_IN", "WON", "OWNS_OR_CONTROLS", "ATTENDED", "AUTHORED", "INVITED", "WAS_COMPLETED_BY", "DISCUSSING", "OFFERS", "OTHER"}

wrong_additions = []

class Entity(BaseModel):
    name: str = Field(description="proper noun for an object of type as below")
    type: Literal["Person", "Organization", "Location", "Project", "Award", "Asset", "Event", "Job", "Meeting", "Document", "Concept"]

class Evidence(BaseModel):
    file_id: Optional[int] = None
    quote: str = Field(description="Exact substring from the file")
    time_stamp: Optional[datetime] = None
    line_number: Optional[int] = None

class smallEvidence(BaseModel):
    quote: str = Field(description="Exact substring from the file")

class ExtractionMeta(BaseModel):
    model: str
    prompt_version: str
    schema_version: str
    extraction_time: datetime

class smallClaim(BaseModel):
    subject_name: str
    predicate: str
    object_name: str
    confidence: float = Field(ge=0, le=1, description="Model confidence between 0 and 1")
    evidences: List[smallEvidence] = Field(default_factory=list, description="all supporting evidence instances")

    @field_validator("predicate", mode="before")
    @classmethod
    def sanitize_predicate(cls, value):
        if not value:
            return "OTHER"
        clean_value = str(value).strip().upper().replace(" ", "_")
        if clean_value in ALLOWED_PREDICATES:
            return clean_value
        else:
            wrong_additions.append(clean_value)
            return "OTHER"

class MemoryExtraction(BaseModel):
    entities: List[Entity] = []
    claims: List[smallClaim] = []

class Claim(BaseModel):
    subject: Entity
    object: Entity
    predicate: Literal["WORKS_ON", "WORKS_UNDER", "CAUSES_ISSUE_IN", "RESOLVED", "NOT_RESOLVED", "MANAGES", "LOCATED_IN", "WON", "OWNS_OR_CONTROLS", "ATTENDED", "AUTHORED", "INVITED", "WAS_COMPLETED_BY", "DISCUSSING", "OFFERS", "OTHER"]
    
    confidence: float
    evidences: List[Evidence]
    metadata: Optional[ExtractionMeta] = None

In [199]:
# the prompt to be given before each text file

prompt = """You are an information extraction system.

Extract structured knowledge from the provided text.

You must return STRICTLY valid JSON that matches the provided schema.

Do not output anything except valid JSON.

CRITICAL RULE: You are strictly forbidden from inventing new predicates. You must ONLY use the exact predicates listed in the schema. If a relationship in the text (like 'offering a class') does not fit one of the allowed predicates, DO NOT extract that claim. Ignore it entirely.

CRITICAL JSON FORMATTING: Do not escape single quotes. Never output \' inside your JSON strings. Just use standard single quotes (') or regular text.
---------------------------------------

TASK:

1. Extract all entities of type:
   - Person
   - Organization
   - Location
   - Project
   - Award
   - Asset
   - Event
   - Job
   - Meeting
   - Document
   - Concept

2. Extract claims between entities using only these predicates:
   - WORKS_ON 
   - DOES_NOT_WORK_ON
   - CAUSES_ISSUE_IN
   - RESOLVED
   - MANAGES
   - LOCATED_IN
   - WON
   - OWNS_OR_CONTROLS
   - ATTENDED
   - AUTHORED
   - INVITED
   - OFFERS

3. For each claim:
   - Provide ALL distinct supporting evidence quotes.
   - Each quote must be an EXACT substring from the text.
   - Do NOT paraphrase.
   - If there is no direct textual support, do NOT create the claim.

4. Assign a confidence score between 0 and 1 for each claim using this rubric:

   0.9–1.0 → Explicitly and clearly stated in text  
   0.7–0.89 → Strongly implied but not word-for-word  
   0.4–0.69 → Moderately inferred  
   0.1–0.39 → Weak inference  

   Confidence must reflect:
   - Explicitness of wording
   - Number of independent evidence quotes
   - Whether interpretation was required

5. The confidence must be a decimal number (e.g., 0.82).
6. Your JSON must exactly match this structure:
    {
      "entities": [
        {"name": "string", "type": "Person | Organization | Location | Project | Concept | Asset | Event | Job | Document"}
      ],
      "claims": [
        {
          "subject_name": "string",
          "predicate": "WORKS_ON | WORKS_FOR | MANAGES | LOCATED_IN | WON | OWNS_OR_CONTROLS | ATTENDED | AUTHORED | COMMUNICATES_WITH | OFFERS | OTHER",
          "object_name": "string",
          "confidence": 0.9,
          "evidences": [{"quote": "exact substring from text"}]
        }
      ]
    }
---------------------------------------

Text:
"""

In [ ]:
from email.utils import parsedate_to_datetime

def extract_timestamp(text):
    for line in text.splitlines():
        if line.startswith("Date:"):
            raw_date = line.replace("Date:", "").strip()
            return parsedate_to_datetime(raw_date)
    return None

In [7]:
current_prompt_version = "1.0"
current_ontology_version = "1.0"

In [192]:
client = Groq(api_key=groq_api_key)

In [56]:
error_logs_file = "logs.txt"
def print_to_logfile(text):
    with open(error_logs_file, "a") as fh:
        fh.write(text)

In [200]:
def multiple_try_extract(text, llm_model = "llama-3.1-8b-instant"):
    """
    Attempting getting structured claims from the data. 
    Retry a few times, if still data is not obtained in the desired format, return None.
    """
    global prompt
    num_attempts = 3
    for _ in range(num_attempts):
        try: 
            response = client.chat.completions.create(
                model = llm_model,
                messages = [
                    {
                        "role": "system",
                        "content": prompt,
                    },
                    {
                        "role": "user",
                        "content": text
                    }
                ],
                response_format = {"type": "json_object"},
                max_tokens = 4096,
                temperature = 0.1,
                frequency_penalty = 0.4,
            )
            raw_json_string = response.choices[0].message.content
            extracted_dict = json.loads(raw_json_string)
            structured_data = MemoryExtraction(**extracted_dict)
            return structured_data
        except Exception as e:
            with open(error_logs_file, 'a') as lf:
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                lf.write(f"[{timestamp}] Extraction failed. Error: {e}\n")
                lf.write(f"Failed Text Input: {text}\n")
                lf.write("-" * 40 + "\n")
            continue
    return None

In [265]:
# generating id for each file and creating a dict from id to file path and vice versa

file_path_dict = {}
path_to_id = {}
id = 1
for root, dirs, files in os.walk(maildir):
    for file in files:
        path = os.path.join(root, file)
        file_path_dict[id] = path
        path_to_id[path] = id
        id += 1

In [266]:
from bs4 import BeautifulSoup
import re

# code for cleaning data before passing to llm

def remove_html(idx):
    """ 
    I will overwrite the file with removed html tags
    """
    path = file_path_dict.get(idx)
    if not path:
        print(f"Warning: File ID {idx} not found in dictionary.")
        return
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
            raw_text = fh.read()
    except Exception as e:
        print(f"Failed to read {path}: {e}")
        return
    if '<html' in raw_text.lower() or '<body' in raw_text.lower() or '<div' in raw_text.lower():
        soup = BeautifulSoup(raw_text, "html.parser")
        clean_text = soup.get_text(separator="\n")
        clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
        try:
            with open(path, 'w', encoding='utf-8') as fh:
                fh.write(clean_text)
            print_to_logfile(f"Successfully removed HTML from {path}\n")
        except Exception as e:
            print_to_logfile(f"Failed to write to {path}: {e}\n")
    else:
        pass

def clean_enron_artifacts(text):
    text = re.sub(r'</?o=[^>]+>', '', text, flags=re.IGNORECASE)
    text = re.sub(r'/o=[^/\s]+(/[^/\s]+)+', '', text, flags=re.IGNORECASE)
    return text

def clean(idx):
    path = file_path_dict.get(idx)
    if not path:
        print(f"Warning: File ID {idx} not found in dictionary.")
        return
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
            raw_text = fh.read()
    except Exception as e:
        print(f"failed to read {path}: {e}")
        return    
    clean_text = clean_enron_artifacts(raw_text)
    try:
        with open(path, 'w', encoding='utf-8') as fh:
            fh.write(clean_text)
        print_to_logfile(f"Successfully removed enron stuff from {path}\n")
    except Exception as e:
        print_to_logfile(f"Failed to write to {path}: {e}\n")

def clean_all():
    for idx in file_path_dict:
        remove_html(idx)
        clean(idx)

In [188]:
clean_all()

In [267]:
# splitting of email into meaningful parts

import hashlib
import re
import copy

def get_content_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def extract_split_content(raw_body):
    """ 
    Uses some cues observed manually
    We could use llm for inferring the cues but it will exceed the free tier limit
    """
    splitters = [
        r"-----Original Message-----",
        r"---------------------- Forwarded by",
        r"___+ Forwarded by",
    ]
    regex_pattern = re.compile("|".join(splitters), flags=re.IGNORECASE)
    
    results = []
    last_end_idx = 0

    for match in regex_pattern.finditer(raw_body):
        match_start_idx = match.start()
        chunk = raw_body[last_end_idx:match_start_idx]
        start_line = raw_body.count('\n', 0, last_end_idx) + 1
        results.append([start_line,chunk.lower()])
        last_end_idx = match.end()
    final_chunk = raw_body[last_end_idx:]
    final_start_line = raw_body.count('\n', 0, last_end_idx) + 1
    results.append([final_start_line, final_chunk.lower()])
    return results

def atomic_texts(idx):
    """" 
    Splits the email into parts by looking for cues that hint forwarding.
    """
    path = file_path_dict[idx]
    
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
            raw_text = fh.read()
    except Exception as e:
        print_to_logfile(f"Failed to read {path}: {e}\n")
        return None
    return extract_split_content(raw_text)

In [268]:
from datasketch import MinHash, MinHashLSH

# for detection of similar but not the same emails 

lsh_index = MinHashLSH(threshold=0.90, num_perm=128)

def get_fuzzy_signature(text, num_perm=128):
    """
    Generates a MinHash signature using 3-character shingles.
    """
    m = MinHash(num_perm=num_perm)
    shingles = [text[i : i+3] for i in range(len(text) - 2)]
    if not shingles:
        m.update(text.encode('utf8'))
        return m
        
    for shingle in shingles:
        m.update(shingle.encode('utf8'))
        
    return m

In [269]:
# deterministic normalization of names

def normalize_entity_name(name):
    name = name.strip().lower()
    name = re.sub(r'[\.\,\'\"\;]+$', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

In [270]:
# the main claim extraction function

def extract_claims_all(current_ontology_version, current_prompt_version, llm_model="llama-3.1-8b-instant"):
    """
    Finds all claims using the llm model specified, and returns a dict of claims.
    """
    global_claims = {}
    hash_dict = {}
    num_iters = 0
    for idx in file_path_dict:
        num_iters += 1
        path = file_path_dict[idx]
        print(f"reading file {path}")
        fh = open(path)
        text = fh.read()
        file_timestamp = extract_timestamp(text)
        fh.close()
        split_parts = atomic_texts(idx)
        if split_parts is None:
            continue
        for st_line, part in split_parts:
            hash_val = get_content_hash(part)
            found = True
            if hash_val in hash_dict:
                claims = copy.deepcopy(hash_dict[hash_val])
            else:
                fuzzy_sig = get_fuzzy_signature(part)
                fuzzy_matches = lsh_index.query(fuzzy_sig)
                if fuzzy_matches:
                    normal_hash = fuzzy_matches[0]
                    claims = copy.deepcopy(hash_dict[normal_hash])
                else:
                    structured = multiple_try_extract(part)
                    if structured is None:
                        print_to_logfile(f"Error occured in inferring from {path} starting at line number {st_line}\n")
                        if num_iters > 100:
                            print(f"Error encountered with inference. Number of iterations so far {num_iters}")
                            print("Do you want to stop now? [Y/N]")
                            c = input().strip().split()[0].lower()
                            if c == 'y':
                                return global_claims, hash_dict.keys()
                        continue
                    claims = structured.claims
                    ents = structured.entities
                    lookup_dict = {e.name: e.type for e in ents}
                    new_claims = []
                    for claim in claims:
                        subj_type = lookup_dict.get(claim.subject_name, "Concept")
                        obj_type =  lookup_dict.get(claim.object_name, "Concept")
                        Full_claim = Claim(
                            subject = Entity(type=subj_type, name=claim.subject_name),
                            object = Entity(type=obj_type, name=claim.object_name),
                            predicate = claim.predicate,
                            confidence = claim.confidence,
                            metadata = None,
                            evidences = []
                        )
                        for ev in claim.evidences:
                            Full_claim.evidences.append(Evidence(quote=ev.quote))
                        new_claims.append(Full_claim)
                    claims = new_claims
                    found = False
                    hash_dict[hash_val] = []
                    lsh_index.insert(hash_val, fuzzy_sig)
            meta = ExtractionMeta(
                model=llm_model,
                prompt_version = current_prompt_version,
                schema_version = current_ontology_version,
                extraction_time = datetime.now()
            )
            print(f"extraction of {path} done")
            for claim in claims:
                # normalizing names for subject and object
                claim.subject.name = normalize_entity_name(claim.subject.name)
                claim.object.name  = normalize_entity_name(claim.object.name)
                # adding file_name and time_stamp to evidences
                new_ev = []
                for ev in claim.evidences:
                    lines = part.splitlines()
                    line_number_searched = -1
                    for line_num in range(len(lines)):
                        if ev.quote in lines[line_num]:
                            line_number_searched = line_num
                            break
                    line_number_searched = (line_number_searched + st_line if line_number_searched != -1 else -1)
                    ev_obj = Evidence(quote=ev.quote, file_id=idx, time_stamp=file_timestamp, line_number=line_number_searched)
                    new_ev.append(ev_obj)
                claim.evidences = new_ev
                claim.metadata = meta
                key = (claim.subject.name, claim.predicate, claim.object.name)
                if key not in global_claims:
                    global_claims[key] = claim
                else:
                    global_claims[key].evidences.extend(claim.evidences)
                    # heuristic combination of confidences
                    global_claims[key].confidence = max(min(global_claims[key].confidence + 0.4*(claim.confidence-0.5), 1), 0)
                    # look for a better strategy for replacement, use time somehow?
                    if claim.metadata.extraction_time > global_claims[key].metadata.extraction_time:
                        global_claims[key].metadata = meta
                if not found:
                    hash_dict[hash_val].append(claim)
        # in case bad events happen, we will atleast have our json
        with open("claims_snapshot.json", "w") as out:
            json.dump(
                {str(k): v.model_dump(mode="json") for k, v in global_claims.items()},
                out,
                indent=2
            )
    return global_claims, hash_dict.keys()

The below cell hasn't been run because it takes a lot of time to run, and because of the print statements inside our code, a big block of printed statements was generated, also at that time i had not put if num_iters > 100 condition inside the code, so i had to manually interrupt the cell and re obtain the claims from the json snapshot. however the below cell is correct and is ready for production use. 

In [ ]:
lsh_index = MinHashLSH(threshold=0.90, num_perm=128)
global_claims, hash_vals = extract_claims_all(current_ontology_version, current_prompt_version)

Free tier API's dont allow many queries, this is as far as i could get on free tier.

In [ ]:
# for actually inserting memory into our database

def upsert_claim_to_neo4j(claim, status: str, driver):
    """
    Upserts a claim into Neo4j. Status can be 'PERMANENT' or 'NEEDS_REVIEW'.
    Entity dedup done later async from this.
    """
    query = """
    MERGE (sub:Entity {name: $sub_name})
    ON CREATE SET sub.type = $sub_type
    
    MERGE (obj:Entity {name: $obj_name})
    ON CREATE SET obj.type = $obj_type
    MERGE (sub)-[:SUBJECT_OF]->(c:Claim {predicate: $predicate})-[:HAS_OBJECT]->(obj)
    ON CREATE SET 
        c.confidence = $confidence, 
        c.status = $status,
        c.schema_version = $schema_version,
        c.prompt_version = $prompt_version,
        c.model = $model,
        c.last_updated_time = datetime($extraction_time) // Set the initial clock
    ON MATCH SET 
        c.confidence = CASE WHEN $confidence > c.confidence THEN $confidence ELSE c.confidence END,
        c.status = CASE WHEN $confidence > 0.7 THEN 'PERMANENT' ELSE c.status END,
        c.last_updated_time = datetime($extraction_time)
    
    WITH c
    UNWIND $evidences as ev
    
    MERGE (e:Evidence {quote: ev.quote})

    MERGE (c)-[:SUPPORTED_BY]->(e)

    MERGE (f:File {id: ev.file_id})

    MERGE (f)-[r:CONTAINS_EVIDENCE]->(e)
    ON CREATE SET 
        r.line_number = ev.line_number_in_file,
        r.time_stamp = CASE WHEN ev.time_stamp IS NOT NULL THEN datetime(ev.time_stamp) ELSE NULL END
    """
    
    evidences_data = [
        {
            "quote": ev.quote, 
            "file_id": ev.file_id, 
            "line_number_in_file": ev.line_number, 
            "time_stamp": ev.time_stamp.isoformat() if ev.time_stamp else None
        } 
        for ev in claim.evidences
    ]
    
    parameters = {
        "sub_name": claim.subject.name,
        "sub_type": claim.subject.type,
        "obj_name": claim.object.name,
        "obj_type": claim.object.type,
        "predicate": claim.predicate,
        "confidence": claim.confidence,
        "status": status,
        "schema_version": claim.metadata.schema_version,
        "prompt_version": claim.metadata.prompt_version,
        "model": claim.metadata.model,
        "extraction_time": claim.metadata.extraction_time.isoformat() if claim.metadata.extraction_time else None,
        "evidences": evidences_data
    }
    
    with driver.session() as session:
        session.execute_write(lambda tx: tx.run(query, parameters))

def process_extracted_claims(global_claims, driver, threshold=0.7):
    for key, claim in global_claims.items():
        if claim.predicate == "OTHER":
            claim.confidence = 0.5
        if claim.confidence > threshold and claim.predicate != "OTHER":
            upsert_claim_to_neo4j(claim, status="PERMANENT", driver=driver)
        elif claim.confidence > 0:
            upsert_claim_to_neo4j(claim, status="NEEDS_REVIEW", driver=driver)

In [272]:
# this is just a helper needed in case something goes wrong with claim extraction and you need to 
# read the snapshot of claims from the saved json

def load_claims_from_snapshot(file_path="claims_snapshot.json"):
    recovered_claims = {}
    try:
        with open(file_path, "r") as fh:
            raw_data = json.load(fh)
            
        for string_key, claim_dict in raw_data.items():
            claim_obj = Claim.model_validate(claim_dict)
            key = (claim_obj.subject.name, claim_obj.predicate, claim_obj.object.name)
            recovered_claims[key] = claim_obj
            
        print(f"Successfully recovered {len(recovered_claims)} claims from {file_path}.")
        return recovered_claims
        
    except FileNotFoundError:
        print(f"Error: Could not find '{file_path}'. Make sure you are in the correct directory.")
        return {}
    except Exception as e:
        print(f"An error occurred while parsing: {e}")
        return {}

In [208]:
global_claims = load_claims_from_snapshot("claims_snapshot.json")

Successfully recovered 724 claims from claims_snapshot.json.


In [291]:
# the below cell just calls the function defined above

In [215]:
process_extracted_claims(global_claims, driver)

Note that "NEEDS_REVIEW" memory is not to be trusted. I find it pretty bad.

In [275]:
# for handling changes in files, because the redactions and deletions might change files
# this function just changes dictionaries, not the nodes in the graph
# graph updation function is below

def update_file_dictionaries(maildir, file_path_dict, path_to_id):
    """
    Scans the maildir to add new files and remove deleted files from dictionaries.
    """
    current_files_on_disk = set()
    for root, dirs, files in os.walk(maildir):
        for file in files:
            path = os.path.join(root, file)
            current_files_on_disk.add(path)

    paths_in_dict = list(path_to_id.keys())
    for old_path in paths_in_dict:
        if old_path not in current_files_on_disk:
            print(f"File deleted from disk: {old_path}")

            old_id = path_to_id.pop(old_path)
            file_path_dict.pop(old_id, None)

    next_id = max(file_path_dict.keys(), default=0) + 1
    
    for new_path in current_files_on_disk:
        if new_path not in path_to_id:

            file_path_dict[next_id] = new_path
            path_to_id[new_path] = next_id
            next_id += 1
            
    print(f"Sync complete. Tracking {len(file_path_dict)} active files.")
    return file_path_dict, path_to_id

In [276]:
# if a file is reprocessed, there is no point in keeping the previous processing as a evidence
# because old evidence might just be false at this point

def wipe_file_footprint(file_id, driver):
    """
    Deletes all evidence for a specific file. 
    If a claim is left with no evidence after this, the claim is deleted.
    """
    query = """
    MATCH (e:Evidence {file_id: $file_id})
    
    OPTIONAL MATCH (c:Claim)-[r:SUPPORTED_BY]->(e)
    DELETE r, e

    WITH c
    WHERE c IS NOT NULL AND NOT (c)-[:SUPPORTED_BY]->()
    DETACH DELETE c
    """
    with driver.session() as session:
        session.run(query, file_id=file_id)

In [277]:
# function for handling changes in the graph
# by default looks for changes in schema/prompt/model
# but allows custom queries allowing it to be used for general changes


def extract_claims_partial(current_ontology_version, current_prompt_version, driver, llm_model="llama3", query = None, query_params = None, empty_lsh = False):
    """
    returns claims as per the new schema for the claims whose ontology_version/schema_version/model don't match current values.
    Its also possible to run this function with a custom query for handling changes in files/ specific needs.
    """
    if empty_lsh:
        # to be done when we want a new lsh_index for the entire dataset if there ae sufficient changes
        global lsh_index
        lsh_index = MinHashLSH(threshold=0.90, num_perm=128)
    params = None
    if query is None:       
        query = """
        MATCH (c:Claim)-[:SUPPORTED_BY]->(e:Evidence)
        WHERE (c.schema_version <> $target_schema OR c.schema_version IS NULL)
           OR (c.prompt_version <> $target_prompt OR c.prompt_version IS NULL)
           OR (c.model <> $target_model OR c.model IS NULL)
        RETURN DISTINCT e.file_id AS file_id
        """
        params = {
            "target_schema": current_ontology_version,
            "target_prompt": current_prompt_version,
            "target_model": llm_model
        }
    else:
        params = query_params
    outdated_files = []
    with driver.session() as session:
        result = session.run(query, params)
        outdated_files = [record["file_id"] for record in result]
    if not outdated_files:
        print("No outdated files found. Database is up to date.")
        return {}
    print(f"Found {len(outdated_files)} files requiring reprocessing.")
    global_claims = {}
    hash_dict = {}
    num_iters = 0
    for idx in outdated_files:
        num_iters += 1
        path = file_path_dict.get(idx, None)
        if path is None:
            print(f"Warning: File {path} no longer exists.")
            continue
        fh = open(path)
        text = fh.read()
        fh.close()
        wipe_file_footprint(idx, driver)
        if text == "":
            continue
        file_timestamp = extract_timestamp(text)
        split_parts = atomic_texts(idx)
        if split_parts is None:
            continue
        for st_line, part in split_parts:
            hash_val = get_content_hash(part)
            if hash_val in hash_dict:
                claims = copy.deepcopy(hash_dict[hash_val])
            else:
                fuzzy_sig = get_fuzzy_signature(part)
                fuzzy_matches = lsh_index.query(fuzzy_sig)
                if fuzzy_matches:
                    normal_hash = fuzzy_matches[0]
                    claims = copy.deepcopy(hash_dict[normal_hash])
                else:
                    structured = multiple_try_extract(part)
                    if structured is None:
                        if num_iters > 100:
                            print(f"Error encountered with inference. Number of iterations so far {num_iters}")
                            print("Do you want to stop now? [Y/N]")
                            c = input().strip().split()[0].lower()
                            if c == 'y':
                                return global_claims, hash_dict.keys()
                        continue
                    claims = structured.claims
                    hash_dict[hash_val] = []
                    lsh_index.insert(hash_val, fuzzy_sig)
            meta = ExtractionMeta(
                model=llm_model,
                prompt_version = current_prompt_version,
                schema_version = current_ontology_version,
                extraction_time = datetime.now()
            )
            for claim in claims:
                # normalizing names for subject and object
                claim.subject.name = normalize_entity_name(claim.subject.name)
                claim.object.name  = normalize_entity_name(claim.object.name)
                # adding file_name and time_stamp to evidences
                new_ev = []
                for ev in claim.evidences:
                    lines = part.splitlines()
                    line_number_searched = -1
                    for line_num in range(len(lines)):
                        if ev.quote in lines[line_num]:
                            line_number_searched = line_num
                            break
                    line_number_searched = (line_number_searched+st_line if line_number_searched != -1 else -1)
                    ev_obj = Evidence(quote=ev.quote, file_id=idx, time_stamp=file_timestamp, line_number=line_number_searched)
                    new_ev.append(ev_obj)
                claim.evidences = new_ev
                claim.metadata = meta
                key = (claim.subject.name, claim.predicate, claim.object.name)
                if key not in global_claims:
                    global_claims[key] = claim
                else:
                    global_claims[key].evidences.extend(claim.evidences)
                    # heuristic combination of confidences
                    global_claims[key].confidence = max(min(global_claims[key].confidence + 0.4*(claim.confidence-0.5), 1), 0)
                    # look for a better strategy for replacement, use time somehow?
                    if claim.metadata.extraction_time > global_claims[key].metadata.extraction_time:
                        global_claims[key].metadata = meta
                if hash_val not in hash_dict:
                    hash_dict[hash_val].append(claim)
            
    process_extracted_claims(global_claims, driver)
    
    return global_claims, hash_dict.keys()

In [278]:
import pickle

# we need to save lsh and hash_vals for fuzzy matching, so we give functions to save it to a file for use later

STATE_FILE = "dedup_state.pkl"

def load_dedup_state():
    """Loads the exact-match hashes and fuzzy LSH index from disk."""
    if os.path.exists(STATE_FILE):
        print("Loading previous deduplication memory from disk...")
        with open(STATE_FILE, 'rb') as f:
            state = pickle.load(f)
            return state['lsh_index'], state['hash_vals']
    else:
        print("No previous memory found. Starting with empty indexes.")
        # Return an empty set and a fresh LSH index
        return set(), MinHashLSH(threshold=0.90, num_perm=128)

def save_dedup_state(lsh_index):
    """Saves the current state of both indexes to disk for the next run."""
    with open(STATE_FILE, 'wb') as f:
        pickle.dump({
            'lsh_index': lsh_index,
            'hash_vals': hash_vals
        }, f)
    print("Saved deduplication memory to disk")

In [218]:
save_dedup_state(lsh_index)

Saved deduplication memory to disk


In [279]:
# reduce confidence and change state from permanent to needs_review if memory is too old

def decay(driver, reduction_factor=0.85, threshold=0.7):
    """ 
    Reduces the confidence of PERMANENT claims by a factor if they haven't been 
    changed in the last 6 months.
    Downgrades to NEEDS_REVIEW if confidence drops below the threshold.
    """
    query = """
    MATCH (c:Claim {status: 'PERMANENT'})
    
    WHERE coalesce(c.last_decayed, c.last_updated_time) < datetime() - duration({months: 6})

    SET c.confidence = c.confidence * $reduction_factor

    SET c.status = CASE 
        WHEN c.confidence < $threshold THEN 'NEEDS_REVIEW' 
        ELSE c.status 
    END

    SET c.last_decayed = datetime()
    
    WITH c, CASE WHEN c.status = 'NEEDS_REVIEW' THEN 1 ELSE 0 END AS downgraded
    RETURN count(c) AS total_decayed, sum(downgraded) AS total_downgraded
    """
    
    parameters = {
        "reduction_factor": reduction_factor,
        "threshold": threshold
    }
    
    try:
        with driver.session() as session:
            result = session.execute_write(lambda tx: tx.run(query, parameters).single())
            total_decayed = result["total_decayed"] if result and result["total_decayed"] else 0
            total_downgraded = result["total_downgraded"] if result and result["total_downgraded"] else 0
            
        print(f"Decay complete: {total_decayed} claims decayed.")
        print(f"Triggered NEEDS_REVIEW for {total_downgraded} claims.")
        
        return total_decayed, total_downgraded
        
    except Exception as e:
        print(f"Failed to execute decay: {e}")
        return 0, 0

In [280]:
# if a claim is kind of unique like
# A is CEO of B, then we might want to delete older memory of the form C is CEO of B
# this function retrieves the latest memory, and optionally allows for deletion of older memory

def get_and_fix_uinque(driver, predicate, remove=False):
    """
    Queries the driver to find all claims with given predicate,
    returns the latest one, and removes all other claims if remove is True
    """
    fetch_query = """
    MATCH (sub:Entity)-[:SUBJECT_OF]->(c:Claim {predicate: $predicate})-[:HAS_OBJECT]->(obj:Entity)
    
    OPTIONAL MATCH (c)-[:SUPPORTED_BY]->(e:Evidence)
    WITH c, sub, obj, max(e.time_stamp) AS latest_date
    ORDER BY latest_date DESC
    
    WITH collect({
        claim_id: id(c), 
        subject: sub.name, 
        object: obj.name, 
        latest_date: latest_date
    }) AS all_claims

    WHERE size(all_claims) > 0
    
    WITH all_claims[0] AS latest, all_claims[1..] AS others
    RETURN latest, [x IN others | x.claim_id] AS obsolete_ids
    """
    
    delete_query = """
    MATCH (c:Claim)
    WHERE id(c) IN $obsolete_ids
    DETACH DELETE c
    """
    
    with driver.session() as session:
        result = session.execute_read(lambda tx: tx.run(fetch_query, {"predicate": predicate}).single())
        
        if not result:
            print(f"No claims found for predicate: '{predicate}'.")
            return None
            
        latest_claim = result["latest"]
        obsolete_ids = result["obsolete_ids"]
        
        if remove and obsolete_ids:
            session.execute_write(lambda tx: tx.run(delete_query, {"obsolete_ids": obsolete_ids}))
            print(f"Cleaned up graph: Removed {len(obsolete_ids)} older claims for predicate '{predicate}'.")
            
        return latest_claim

# Dedup and cannonicalization handling

In [281]:
# there is no specific reason for using gemini below, i just ran out of free tier llama-3

In [226]:
from google import genai
cl1 = genai.Client(api_key = gemini_api_key)

In [228]:
new_client = instructor.from_genai(cl1)

In [282]:
# Asynchronous entity cannonicalization
# Here we look for similarity in names and works of entitities to see if they are actually the same
# create canonical nodes if entries are the same

class isSame(BaseModel):
    is_same : bool
    canonical_name : str

def get_entity_context(entity_id, driver):
    """
    Pulls the surrounding graph neighborhood for an entity to give the LLM context.
    Finds what predicates and other entities this entity is connected to.
    """
    query = """
    MATCH (e:Entity) WHERE elementId(e) = $entity_id
    OPTIONAL MATCH (e)-[:SUBJECT_OF]->(c:Claim)-[:HAS_OBJECT]->(obj:Entity)
    OPTIONAL MATCH (sub:Entity)-[:SUBJECT_OF]->(c2:Claim)-[:HAS_OBJECT]->(e)

    WITH e, 
         collect(DISTINCT {role: "subject", action: c.predicate, target: obj.name})[..5] AS actions_done,
         collect(DISTINCT {role: "object", action: c2.predicate, source: sub.name})[..5] AS actions_received
    
    RETURN e.name AS name, e.type AS type, actions_done, actions_received
    """
    with driver.session() as session:
        result = session.run(query, {"entity_id": entity_id}).single()
        return result.data() if result else None

def get_canonical_entity_context(canonical_id, driver):
    query = """
    MATCH (canon:CanonicalEntity) WHERE elementId(canon) = $canonical_id
    OPTIONAL MATCH (alias:Entity)-[:RESOLVES_TO]->(canon)

    OPTIONAL MATCH (alias)-[:SUBJECT_OF]->(c_out:Claim)-[:HAS_OBJECT]->(obj:Entity)
    OPTIONAL MATCH (sub:Entity)-[:SUBJECT_OF]->(c_in:Claim)-[:HAS_OBJECT]->(alias)
    WITH canon,
         [x IN collect(DISTINCT {role: "subject", action: c_out.predicate, target: obj.name}) WHERE x.action IS NOT NULL][..5] AS actions_done,
         [x IN collect(DISTINCT {role: "object", action: c_in.predicate, source: sub.name}) WHERE x.action IS NOT NULL][..5] AS actions_received

    RETURN canon.name AS name, canon.type AS type, actions_done, actions_received
    """
    with driver.session() as session:
        result = session.run(query, {"canonical_id": canonical_id}).single()
        return result.data() if result else None

def evaluate_merge_with_llm(context1, context2, llm_model = "gemini-2.5-flash"):
    """
    Prompts the LLM to decide if two entities are the same real-world thing.
    """
    prompt = f"""
    You are a Knowledge Graph Entity Resolution bot. 
    Look at these two entities extracted from the Enron email corpus and their graph context.
    
    Entity 1: "{context1['name']}" (Type: {context1['type']})
    Context: {context1['actions_done']} | {context1['actions_received']}
    
    Entity 2: "{context2['name']}" (Type: {context2['type']})
    Context: {context2['actions_done']} | {context2['actions_received']}
    
    Are these referring to the exact same real-world person, organization, or project?
    Return a JSON object with two keys:
    - "is_same": boolean (true or false)
    - "canonical_name": string (The best, most complete name to use for them. If false, return "")
    """
    
    try: 
        response = new_client.chat.completions.create(
            model = llm_model,
            response_model = isSame,
            max_retries = 4,
            messages = [
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
        )
        return response
    except Exception as e:
        return isSame(is_same=False, canonical_name="")
    
def async_entity_dedup(driver):
    """
    finds similar named entities and merges them.
    might need to run multiple times because limits are set in queries
    """
    candidate_query_1 = """
    MATCH (e:Entity), (canon:CanonicalEntity)
    WHERE NOT (e)-[:RESOLVES_TO]->()
      AND e.type = canon.type
      AND (
        apoc.text.distance(e.name, canon.name) <= 2
        OR
        (size(e.name) > 4 AND size(canon.name) > 4 AND (e.name CONTAINS canon.name OR canon.name CONTAINS e.name))
        OR
        apoc.text.sorensenDiceSimilarity(e.name, canon.name) > 0.8
        )
      
    RETURN elementId(e) AS e_id, elementId(canon) AS canon_id
    LIMIT 100
    """

    candidate_query_2 = """
    MATCH (e1:Entity), (e2:Entity)
    WHERE elementId(e1) < elementId(e2) 
      AND e1.type = e2.type
      AND NOT (e1)-[:RESOLVES_TO]->(:CanonicalEntity)
      AND NOT (e2)-[:RESOLVES_TO]->(:CanonicalEntity)
      AND (
        apoc.text.distance(e1.name, e2.name) <= 2
        OR
        (size(e1.name) > 4 AND size(e2.name) > 4 AND (e1.name CONTAINS e2.name OR e2.name CONTAINS e1.name))
        OR
        apoc.text.sorensenDiceSimilarity(e1.name, e2.name) > 0.8
        )
    RETURN elementId(e1) AS id1, elementId(e2) AS id2
    LIMIT 100
    """

    merge_query_1 = """
    MATCH (e:Entity) WHERE elementId(e) = $e_id
    MATCH (canon:CanonicalEntity) WHERE elementId(canon) = $canon_id
    
    MERGE (e)-[r:RESOLVES_TO]->(canon)
    ON CREATE SET 
        r.timestamp = datetime(),
        r.model = "llama-3.1-8b-instant",
        r.reason = $reason
    """

    merge_query_2 = """
    MERGE (canon:CanonicalEntity {name: $canonical_name})
    ON CREATE SET canon.type = $entity_type
    
    WITH canon
    MATCH (e1:Entity) WHERE elementId(e1) = $id1
    MATCH (e2:Entity) WHERE elementId(e2) = $id2
    
    MERGE (e1)-[r1:RESOLVES_TO]->(canon)
    ON CREATE SET r1.timestamp = datetime(), r1.reason = $reason, r1.model = "llama-3.1-8b-instant"
    
    MERGE (e2)-[r2:RESOLVES_TO]->(canon)
    ON CREATE SET r2.timestamp = datetime(), r2.reason = $reason, r2.model = "llama-3.1-8b-instant"
    """

    with driver.session() as session:
        candidates = session.run(candidate_query_1).data()
        for pair in candidates:
            #print_to_logfile(pair)
            #print_to_logfile('\n')
            ctx_e = get_entity_context(pair['e_id'], driver)
            ctx_canon = get_canonical_entity_context(pair['canon_id'], driver)
            
            decision = evaluate_merge_with_llm(ctx_e, ctx_canon)
            
            if decision.is_same:
                session.execute_write(lambda tx: tx.run(merge_query_1, {
                    "e_id": pair['e_id'], 
                    "canon_id": pair['canon_id'], 
                    "reason": "LLM matched to existing canonical",
                }))
    
    with driver.session() as session:
        candidates = session.run(candidate_query_2).data()
        for pair in candidates:
            ctx1 = get_entity_context(pair['id1'], driver)
            ctx2 = get_entity_context(pair['id2'], driver)
            decision = evaluate_merge_with_llm(ctx1, ctx2)
            
            if decision.is_same:
                session.execute_write(lambda tx: tx.run(merge_query_2, {
                    "canonical_name": decision.canonical_name.lower(),
                    "entity_type": ctx1["type"],
                    "id1": pair['id1'], 
                    "id2": pair['id2'],
                    "reason": "LLM created new canonical match",
                }))

In [262]:
async_entity_dedup(driver)

In [283]:
# this function allows for undoing merges
# note that above functions write a reason for merging
# we might wanna be specific there
# also we might wanna consult that reason before undoing merges

def undo_entity_merge(entity_id, driver):
    query = """
    MATCH (e:Entity)-[r:RESOLVES_TO]->(canon:CanonicalEntity)
    WHERE id(e) = $entity_id
    DELETE r
    WITH canon
    WHERE size((canon)<-[:RESOLVES_TO]-()) = 0 
    DELETE canon 
    RETURN "Success" as status
    """
    with driver.session() as session:
        session.execute_write(lambda tx: tx.run(query, {
            "entity_id": entity_id,
        }))

# Extra work for submission

In [289]:
def serialize_to_cypher(driver, file):
    export_query = """
    CALL apoc.export.cypher.all(null, {
        format: 'cypher-shell',
        stream: true
    })
    YIELD cypherStatements
    RETURN cypherStatements
    """
    
    with driver.session() as session:
        result = session.run(export_query)
        record = result.single()
        
        if record:
            with open(file, "w") as f:
                f.write(record["cypherStatements"])
            print(f"Graph successfully serialized to {file}")

    driver.close()

In [290]:
cypher_file = "graph.cypher"
serialize_to_cypher(driver, cypher_file)

Graph successfully serialized to graph.cypher


In [ ]:
query1 = """
MATCH (canon:CanonicalEntity)
WITH canon, rand() AS r
ORDER BY r
LIMIT 1
MATCH (alias:Entity)-[:RESOLVES_TO]->(canon)

OPTIONAL MATCH (alias)-[:SUBJECT_OF]->(outClaim:Claim {status: 'PERMANENT'})-[:HAS_OBJECT]->(target:Entity)
OPTIONAL MATCH (source:Entity)-[:SUBJECT_OF]->(inClaim:Claim {status: 'PERMANENT'})-[:HAS_OBJECT]->(alias)

RETURN 
    canon.name AS canonical_name,
    canon.type AS type,
    [rel IN collect(DISTINCT {
        predicate: outClaim.predicate, 
        to: target.name, 
        confidence: outClaim.confidence
    }) WHERE rel.predicate IS NOT NULL] AS outgoing_claims,
    [rel IN collect(DISTINCT {
        predicate: inClaim.predicate, 
        from: source.name, 
        confidence: inClaim.confidence
    }) WHERE rel.predicate IS NOT NULL] AS incoming_claims
"""
with driver.session() as session:
    result = session.run(query1).single().data()
print(result)

{'canonical_name': 'brian redmond', 'type': 'Person', 'outgoing_claims': [{'to': 'daily lunches', 'predicate': 'ATTENDED', 'confidence': 0.9}, {'to': 'enron', 'predicate': 'LOCATED_IN', 'confidence': 0.9}, {'to': 'enron', 'predicate': 'MANAGES', 'confidence': 0.9}, {'to': 'mark russ', 'predicate': 'MANAGES', 'confidence': 0.9}, {'to': 'producer/wellhead group (texas)', 'predicate': 'MANAGES', 'confidence': 0.9}], 'incoming_claims': [{'predicate': 'INVITED', 'confidence': 0.9, 'from': 'louise kitchen'}, {'predicate': 'INVITED', 'confidence': 1.0, 'from': 'louise'}]}


In [298]:
query2 = """ 
MATCH (sub:Entity)-[:SUBJECT_OF]->(c:Claim {status: 'NEEDS_REVIEW'})-[:HAS_OBJECT]->(obj:Entity)
WHERE c.predicate IS NOT NULL 
  AND sub.name IS NOT NULL 
  AND obj.name IS NOT NULL

MATCH (c)-[:SUPPORTED_BY]->(ev:Evidence)
WHERE ev.quote IS NOT NULL
WITH sub, c, obj, count(ev) AS evidence_count

WHERE evidence_count > 2 AND c.confidence < 0.7

RETURN 
    sub.name AS subject, 
    c.predicate AS predicate, 
    obj.name AS object, 
    round(c.confidence, 2) AS current_confidence, 
    evidence_count AS total_sources
ORDER BY evidence_count DESC
"""
with driver.session() as session:
    result = session.run(query2).single().data()
print(result)

AttributeError: 'NoneType' object has no attribute 'data'

In [300]:
query3 = """ 
MATCH (e:Entity)-[:SUBJECT_OF]->(c:Claim {status: 'PERMANENT'})
WHERE e.name IS NOT NULL 
  AND c.predicate IS NOT NULL

WITH e, count(DISTINCT c) AS claim_count, collect(DISTINCT c.predicate) AS activities
WHERE claim_count >= 3
RETURN 
    e.name AS entity_name, 
    e.type AS type, 
    claim_count AS total_permanent_claims,
    activities AS types_of_relations
ORDER BY total_permanent_claims DESC
LIMIT 10
"""
with driver.session() as session:
    result = session.run(query3).single().data()
print(result)

{'entity_name': 'enron', 'type': 'Organization', 'total_permanent_claims': 76, 'types_of_relations': ['OFFERS', 'LOCATED_IN', 'MANAGES', 'OWNS_OR_CONTROLS', 'WON', 'CAUSES_ISSUE_IN']}


/home/aditya/.local/lib/python3.10/site-packages/neo4j/_sync/work/result.py:635: UserWarning: Expected a result with a single record, but found multiple.
  warn(
